This project demonstrates an end-to-end geospatial pipeline that connects administrative boundary data with satellite-based change detection to assess vegetation loss across Canada's fire-prone province.


## Environment Setup 
Importing the core geospatial stack,  vector handling (geopandas, shapely), interactive mapping (folium, geemap), and classification/visualization tools. geemap's presence signals this notebook bridges local vector analysis with Google Earth Engine raster processing.

In [ ]:
import sys
!{sys.executable} -m pip install --ignore-installed earthengine-api geemap pyproj


In [ ]:
pip install "folium>=0.12" matplotlib mapclassify

In [ ]:
import geopandas 
import geodatasets
import pandas as pd
import os
import folium
import matplotlib.pyplot as plt
import mapclassify
import geemap
import webbrowser
from shapely.geometry import mapping

## Load Boundary Data 
Reading the StatCan Census Subdivision boundary shapefile directly from a zip archive using GeoPandas


In [ ]:
home_dir = os.path.expanduser("~")
file_path = f"zip://{home_dir}/path to local file/lcsd000a25a_e.zip"

canada = geopandas.read_file(file_path)

### Data Link - https://www12.statcan.gc.ca/census-recensement/2011/geo/bound-limit/bound-limit-s-eng.cfm?year=25 

## Inspect Schema 
Checking the dataset's column structure to confirm the administrative hierarchy is intact: Province → Census Division → Census Subdivision, each with a paired code/name field, plus the geometry column.

In [ ]:
canada.dtypes


## Full Data Preview 
Viewing the complete DataFrame to confirm row count, column order, and that geometries are populating correctly as polygons.

In [ ]:
canada.describe


## Cardinality 
Check Counting unique values per categorical column as a data-integrity check, validating expected counts (13 provinces, 290 divisions, 5,054 subdivisions) before any filtering begins.

In [ ]:
canada.select_dtypes(include=['object','str']).nunique()

## Baseline Boundary Plot 
A simple outline plot of every subdivision boundary nationwide | a quick visual QA pass before deeper analysis.

In [ ]:
canada.boundary.plot()

## Province-Level  Choropleth 
Coloring subdivisions by parent province to confirm the province groupings render correctly and cover the full country.

In [ ]:

canada.plot(column = 'PRNAME', legend = True, figsize=(15, 10),legend_kwds={'bbox_to_anchor': (1.05, 1), 'loc': 'upper left'})

## Interactive National Map 
Rendering the full dataset interactively via Folium for pan/zoom/attribute inspection.

In [ ]:
canada.explore()

## Named City Lookup 
Filtering to four major Canadian cities by CSDNAME to confirm name-based lookups resolve to the correct, single polygon per city.

In [ ]:
selected_cities = canada[canada['CSDNAME'].isin(['Calgary', 'Vancouver', 'Ottawa', 'Toronto'])]

print(selected_cities)

## City Lookup Results 
Reviewing the matched records to confirm correct province/division metadata for each selected city.

In [ ]:
area = 'greater vancouver' 
region = canada[canada['CDUID'] == '5915']

In [ ]:

#Plot
ax = region.plot(figsize=(8, 8), color='skyblue', edgecolor='black')
ax.set_title(f"{area}  Boundary", fontsize=14)
ax.set_axis_off()  # Hide axis coordinates

## Isolate a Single Census Division 
Narrowing scope to one Census Division (Division No. 6, Alberta  home to Calgary) and rendering it standalone as the first region-level boundary map.

In [ ]:


# 1. Plot the base map
ax = region.plot(figsize=(12, 12), color='skyblue', edgecolor='white', linewidth=0.8)

# 2. Iterate through each row and label the centroid
for idx, row in region.iterrows():
    # Calculate the center point of the polygon
    centroid = row['geometry'].centroid
    
    # Place text at the centroid coordinates
    ax.annotate(
        text=row['CSDNAME'],
        xy=(centroid.x, centroid.y),
        ha='center',            # Horizontal alignment
        va='center',            # Vertical alignment
        fontsize=8,
        fontweight='bold',
        color='black'
    )

ax.set_title(f"Census Boundry in {area}", fontsize=14)
ax.set_axis_off()

## Interactive Regional Map 
Reproducing the same regional boundary as an interactive, tooltip-enabled map for closer inspectio

In [ ]:
# Interactive map with CSDNAME 
region.explore(
    column="CSDNAME",
    tooltip="CSDNAME",       
    legend=False,
    #tiles="CartoDB positron"
)

## Build a Composite Shoreline AOI (area of interest) 
Manually curating a list of Census Divisions that border Lake Ontario and filtering the dataset to this multi-division area of interest, a judgment-based AOI rather than a geometrically derived one.


In [ ]:
lake_front_cds = [
    "Toronto", "Peel", "Halton", "Hamilton", 
    "Niagara", "Durham", "Northumberland", "Prince Edward"
]

# Create a subset of the data
lake_ontario_shoreline = canada[
    (canada['PRNAME'].str.contains("Ontario")) & 
    (canada['CDNAME'].isin(lake_front_cds))
]



lake_ontario = lake_ontario_shoreline.plot(figsize=(12, 12), color='skyblue', edgecolor='blue', linewidth=0.8)

# 2. Iterate through each row and label the centroid
for idx, row in lake_ontario_shoreline.iterrows():
    # Calculate the center point of the polygon
    centroid = row['geometry'].centroid
    
    # Place text at the centroid coordinates
    lake_ontario.annotate(
        text=row['CSDNAME'],
        xy=(centroid.x, centroid.y),
        ha='center',            # Horizontal alignment
        va='center',            # Vertical alignment
        fontsize=8,
        fontweight='bold',
        color='black'
    )

lake_ontario.set_title(f"Census Boundry in lake Ontario/America", fontsize=14)
lake_ontario.set_axis_off()

## Interactive Shoreline Map 
Rendering the composite Lake Ontario AOI interactively to visually confirm the selected divisions form a contiguous, sensible boundary.

In [ ]:
lake_ontario_shoreline.explore()

# Change Detection Analysis | Vegetation Loss

## Earth Engine Authentication 
Connecting the notebook to Google Earth Engine and confirming the session is live the bridge point between local vector work and cloud-based raster analysis.


In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='#google project id')
print(ee.String('Hello from the Earth Engine servers!').getInfo())

## Load Boundary Layer into Earth Engine 
Re-importing the same StatCan boundary layer as a server-side FeatureCollection asset, required for clipping and reducing raster data natively within GEE.

In [ ]:
raw_canada = ee.FeatureCollection("#path to server file")


## Filter to Fire-Prone Provinces (Local) 
Narrowing the national boundary dataset to six provinces/territories with well-documented wildfire exposure, using their PRUID codes.

In [ ]:
fire_prone_pr = canada[canada['PRUID'] .isin (['59','48','47','24','61','60'])]
fire_prone_pr[['PRUID','PRNAME']].drop_duplicates()

## Validate Filter Server-Side 
Reapplying the same six-province filter directly on the Earth Engine FeatureCollection and cross-checking the result against the local filter for consistency.

In [ ]:
#Define your target PRUID list exactly
target_pruids = ee.List(['59', '48', '47', '24', '61', '60'])

# Filter the cloud collection to ONLY your focus zones
focused_collection = raw_canada.filter(ee.Filter.inList('PRUID', target_pruids))
# Filter Earth Engine FeatureCollection to a single province — e.g., British Columbia (PRUID '59')
single_province = raw_canada.filter(ee.Filter.eq('PRUID', '59'))


#Group by PRUID and PRNAME to count the focus municipalities on the server
# (This extracts just the text attributes, keeping it lightning fast)
summary_features = focused_collection.reduceColumns(
    reducer=ee.Reducer.toList().repeat(2),
    selectors=['PRUID', 'PRNAME']
).getInfo()

extracted_list = summary_features['list']
#  Build a clean Pandas DataFrame from the server output
df_focus = pd.DataFrame({
    'PRUID': extracted_list[0],
    'PRNAME': extracted_list[1]
})
# Drop duplicates to show the distinct focus areas and sort them cleanly
df_focus_summary = df_focus.drop_duplicates().sort_values(by='PRUID').reset_index(drop=True)

df_focus_summary


## Finalize Fire Zone AOI 
Confirming canada_fire_zones as the authoritative Earth Engine boundary layer that all subsequent raster operations will be scoped to.

In [ ]:

canada_fire_zones = raw_canada.filter(ee.Filter.inList('PRUID', target_pruids))


## Define the NBR Function 
Writing a reusable function to calculate the Normalized Burn Ratio from NIR (Near-Infrared) and SWIR2 (Shortwave-Infrared) bands, ensuring identical methodology is applied to every composite that follows.

In [ ]:
# Function to calculate NBR using Landsat 8/9 Tier 1 Surface Reflectance
# Band 5 = Near-Infrared (NIR), Band 7 = Shortwave-Infrared 2 (SWIR2)
def calculate_nbr(image):
    return image.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')


## Build the 2022 Pre-Fire Baseline 
Compositing cloud-filtered Landsat 8 imagery from July–August 2022 and calculating its NBR |  the "before" state for change detection.

In [ ]:
# A. Summer 2022 Baseline Composite (Pre-Fire / Past State)
landsat_2022 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                 .filterDate('2022-07-01', '2022-08-31') \
                 .filter(ee.Filter.lt('CLOUD_COVER', 15)) \
                 .median()
nbr_2022 = calculate_nbr(landsat_2022)

## Build the 2025 Post-Fire Composite 
Repeating the identical compositing and NBR process for July–August 2025 |  the "after" state, using the same seasonal window and cloud threshold as the baseline.

In [ ]:
# B. Summer 2025 Condition Composite (Post-Fire / Present State)
landsat_2025 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                 .filterDate('2025-07-01', '2025-08-31') \
                 .filter(ee.Filter.lt('CLOUD_COVER', 15)) \
                 .median()
nbr_2025 = calculate_nbr(landsat_2025)

## Calculate the Change Layer (dNBR) 
Subtracting post-fire NBR from pre-fire NBR to produce a single band quantifying vegetation loss | the core output of the change detection analysis.

In [ ]:
# C. Calculate Change Detection (Delta NBR)
# Pre-Fire NBR minus Post-Fire NBR. High values = massive vegetation drop.
dnbr = nbr_2022.subtract(nbr_2025).rename('dNBR')


## Clip to Study Area 
Restricting the dNBR layer and both source composites to the six-province fire zone boundary, tying the raster analysis to the vector AOI established earlier.

In [ ]:
# CLIP IMAGERY TO EXTRACT SPECIFIC STUDY DISTRICTS

dnbr_clipped = dnbr.clip(canada_fire_zones)
landsat_2022_clipped = landsat_2022.clip(canada_fire_zones)
landsat_2025_clipped = landsat_2025.clip(canada_fire_zones)

In [ ]:
# Clip the dNBR layer to just that province
dnbr_single_province = dnbr.clip(single_province)

## Build the Interactive Comparison Map 
Assembling true-color 2022/2025 layers, the dNBR severity layer, and the StatCan boundary outlines into a single interactive map for visual comparison.

In [ ]:
# SPLIT-PANEL INTERACTIVE VISUALIZATION
# Create the base map centered over Western Canada's boreal forest
Map = geemap.Map(center=[54.0, -115.0], zoom=5,tiles='CartoDB positron')

# Set visual rules: True Color (B4=Red, B3=Green, B2=Blue)
true_color_vis = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 30000, 'gamma': 1.4}

# Create Left and Right Tile Layers for the Slider
left_layer = geemap.ee_tile_layer(landsat_2022_clipped, true_color_vis, 'Summer 2022 (Baseline)')
right_layer = geemap.ee_tile_layer(landsat_2025_clipped, true_color_vis, 'Summer 2025 (2025 State)')


# Visual scale for Change Detection: Green = stable, Red/Orange = Heavy Loss
dnbr_vis = {
    'min': -0.1, 
    'max': 0.5, 
    'palette': ['#7a873b', '#ffffff', '#ffcc00', '#ff6600', '#cc0000']
}

# Add the Change Detection layer on top so you can toggle it on/off in the menu
Map.addLayer(dnbr_clipped, dnbr_vis, 'Vegetation Loss Index (dNBR 2022-2025)')

# Draw the StatCan municipal outline vectors right over the top in bright white
boundary_style = canada_fire_zones.style(color='purple', fillColor='00000000', width=1)
Map.addLayer(boundary_style, {'crs': 'EPSG:3347'}, 'StatCan Focus Boundaries')



Single Province

In [ ]:
Map = geemap.Map(center=[54.0, -125.0], zoom=6, tiles='CartoDB positron')  # recenter roughly over BC

dnbr_vis = {
    'min': -0.1, 
    'max': 0.5, 
    'palette': ['#7a873b', '#ffffff', '#ffcc00', '#ff6600', '#cc0000']
}

Map.addLayer(dnbr_single_province, dnbr_vis, 'dNBR — British Columbia')

boundary_style = single_province.style(color='purple', fillColor='00000000', width=1)
Map.addLayer(boundary_style, {}, 'BC Boundary')

In [ ]:
# British Columb Vegetation Loss Analysis (2022 - 2025)
Map

In [ ]:
Analysis by lawrence okolo [linkedin]


## Export for Sharing Saving the interactive map as a standalone HTML file and opening it in-browser 
a portable artifact viewable without Python or GEE credentials.

In [ ]:


# 1. Define where to save the standalone map file (e.g., your local Downloads folder)
download_dir = os.path.join(os.path.expanduser('~'), "Downloads")
html_file = os.path.join(download_dir, "BC_Vegetation_Loss2025_comparison.html")

# 2. Export the live Earth Engine interactive map to HTML
Map.to_html(filename=html_file, title="BC_Vegetation_Loss2025 Change Detection (2022-2025)", width="100%", height="850px")
print(f"Map successfully generated and saved to: {html_file}")

# 3. Automatically pop the map open in a fresh browser tab
webbrowser.open(f"file://{html_file}")